# GeneralRFM and Fisheye Coordinates

GeneralRFM coordinates use provider-backed reference metrics when a named analytic coordinate system is not enough. The fisheye provider maps raw Cartesian-like coordinates to physical Cartesian coordinates through a radial rescaling, then induces the flat reference metric in the raw coordinates.

## Table of Contents

1. [Runtime context and imports](#Runtime-context-and-imports)
2. [Fisheye map construction](#Fisheye-map-construction)
3. [Reference-metric registration](#Reference-metric-registration)
4. [Radius-map samples](#Radius-map-samples)
5. [Metric residuals](#Metric-residuals)
6. [Generated-code support](#Generated-code-support)
7. [Related notebooks](#Related-notebooks)

## Runtime Context and Imports

The default path builds symbolic fisheye objects and inspects the C-code support API. It does not register C functions or write generated files.

In [ ]:
import nrpy
import nrpy.reference_metric as refmetric
import sympy as sp
from nrpy.equations.generalrfm import fisheye
from nrpy.infrastructures.BHaH import generalrfm_precompute
from nrpy.infrastructures.BHaH.fisheye import phys_params_to_fisheye


print("Imported nrpy from:", nrpy.__file__)

required_capabilities = {
    "nrpy.equations.generalrfm.fisheye": (
        fisheye,
        ["GeneralRFMFisheye", "build_fisheye"],
    ),
    "nrpy.reference_metric": (
        refmetric,
        ["ReferenceMetric", "reference_metric"],
    ),
    "nrpy.infrastructures.BHaH.generalrfm_precompute": (
        generalrfm_precompute,
        [
            "register_CFunction_generalrfm_precompute",
            "register_CFunctions_generalrfm_support",
        ],
    ),
    "nrpy.infrastructures.BHaH.fisheye.phys_params_to_fisheye": (
        phys_params_to_fisheye,
        [
            "register_CFunction_fisheye_params_from_physical_N",
            "build_post_params_struct_set_to_default_hook",
        ],
    ),
}

for label, (module, names) in required_capabilities.items():
    print(label)
    for name in names:
        print(" ", name, "available:", hasattr(module, name))

## Fisheye Map Construction

For an `N`-transition fisheye map, the raw radius is transformed by plateau factors `a0` through `aN`, transition centers `R1` through `RN`, transition widths `s1` through `sN`, and a global scale `c`. The physical Cartesian map is radial: each raw coordinate is multiplied by the physical-radius to raw-radius ratio.

In [ ]:
def compact(expr, width=120):
    text = sp.sstr(expr)
    return text if len(text) <= width else text[: width - 3] + "..."


def compact_list(values, width=120):
    return [compact(value, width=width) for value in values]


fisheye_map = fisheye.build_fisheye(2)

print("Transitions:", fisheye_map.num_transitions)
print("Raw coordinates:", fisheye_map.xx)
print("Plateau factors:", fisheye_map.a_list)
print("Transition centers:", fisheye_map.R_list)
print("Transition widths:", fisheye_map.s_list)
print("Scale factor:", fisheye_map.c)
print("xx_to_CartU:")
for component in compact_list(fisheye_map.xx_to_CartU):
    print(" ", component)

## Reference-Metric Registration

Indexing `reference_metric` constructs and caches the `GeneralRFM_fisheyeN2` reference metric. The resulting object records that its provider is `fisheye` and stores provider metadata alongside the reference-metric map.

In [ ]:
rfm = refmetric.reference_metric["GeneralRFM_fisheyeN2"]

print("Reference metric:", rfm.CoordSystem)
print("Provider:", rfm.general_rfm_provider_name)
print("Provider metadata:", rfm.general_rfm_provider_meta)
print("Reference-metric map:")
for component in compact_list(rfm.xx_to_Cart):
    print(" ", component)
print("Reference-metric ghatDD diagonal:", compact_list(rfm.ghatDD[i][i] for i in range(3)))

## Radius-Map Samples

The unscaled radius map is evaluated only at strictly positive raw radii. The substitution dictionary supplies every fisheye parameter used by the two-transition map.

In [ ]:
fisheye_parameter_values = {
    fisheye_map.a_list[0]: sp.Rational(1, 1),
    fisheye_map.a_list[1]: sp.Rational(3, 2),
    fisheye_map.a_list[2]: sp.Rational(2, 1),
    fisheye_map.R_list[0]: sp.Rational(1, 1),
    fisheye_map.R_list[1]: sp.Rational(3, 1),
    fisheye_map.s_list[0]: sp.Rational(2, 5),
    fisheye_map.s_list[1]: sp.Rational(7, 10),
    fisheye_map.c: sp.Rational(1, 1),
}

print("Parameter substitutions:")
for symbol, value in fisheye_parameter_values.items():
    print(" ", symbol, "=", value)

for raw_radius in [
    sp.Rational(1, 4),
    sp.Rational(1, 1),
    sp.Rational(2, 1),
    sp.Rational(4, 1),
]:
    sample_value = fisheye_map.rbar_unscaled.subs(
        {
            fisheye_map.xx[0]: raw_radius,
            fisheye_map.xx[1]: 0,
            fisheye_map.xx[2]: 0,
            **fisheye_parameter_values,
        }
    )
    print(f"r = {raw_radius}: rbar_unscaled = {sp.N(sample_value, 12)}")

## Metric Residuals

The provider metric is induced by the physical Cartesian map. Selected residuals of
$$
\hat{g}_{ij} - \sum_k \frac{\partial X^k}{\partial x^i}
\frac{\partial X^k}{\partial x^j}
$$
simplify to zero.

In [ ]:
selected_pairs = [(0, 0), (0, 1), (1, 1), (2, 2)]
for i, j in selected_pairs:
    induced_metric_ij = sum(
        fisheye_map.dCart_dxxUD[k][i] * fisheye_map.dCart_dxxUD[k][j]
        for k in range(3)
    )
    residual = sp.simplify(fisheye_map.ghatDD[i][j] - induced_metric_ij)
    print(f"residual ({i}, {j}) =", residual)

## Generated-Code Support

BHaH support routines can precompute GeneralRFM quantities and translate physical fisheye parameters to the internal map parameters. Those registrations are part of generated-project setup, not the default symbolic notebook path.

In [ ]:
support_capabilities = [
    (
        "generalrfm_precompute",
        generalrfm_precompute.register_CFunction_generalrfm_precompute,
    ),
    (
        "generalrfm_support",
        generalrfm_precompute.register_CFunctions_generalrfm_support,
    ),
    (
        "fisheye_params_from_physical",
        phys_params_to_fisheye.register_CFunction_fisheye_params_from_physical_N,
    ),
    (
        "post_params_hook",
        phys_params_to_fisheye.build_post_params_struct_set_to_default_hook,
    ),
]

for label, function in support_capabilities:
    print(label, "callable:", callable(function))

## Related Notebooks

[Reference-metric applications](reference_metric_applications.ipynb) covers standard named coordinate systems and precompute-ready factors. [Multicoordinate wave project](../3-wave_equation/wave_equation_multicoordinates.ipynb) shows how generated projects combine coordinate systems in one BHaH executable.